In [1]:
import torch
import os 
import torch.nn as nn
import numpy as np
from torch.nn.utils import clip_grad_norm

In [2]:
class Dictionary(object) : 
    def __init__(self) : 
        self.word2idx = {}
        self.idx2word = {}
        self.idx = 0

    def add_word(self,word) : # to add word to above dicts.
        if word not in self.word2idx : 
            self.word2idx[word] = self.idx
            self.idx2word[self.idx] = word
            self.idx += 1

    def __len__(self) : 
        return len(self.word2idx)

In [3]:
class txtprocess(object) : 
    def __init__(self) : 
        self.dictionary = Dictionary()

    def get_data(self,path,batch_size=20) : 
        with open(path, "r") as f :
            tokens = 0
            for line in f : 
                words = line.split() + ["<eos>"]
                tokens += len(words)
                for word in words : 
                    self.dictionary.add_word(word)
        # in this func, till here, we just added every word of dataset to our custom dict class !
        # now need to create a tensor that contains the index of all the words in the file.
        rep_tensor = torch.LongTensor(tokens)
        index = 0 
        with open(path, "r") as f : 
            for line in f : 
                words = line.split() + ["<eos>"]
                for word in words : 
                    rep_tensor[index] = self.dictionary.word2idx[word]
                    index += 1
        num_batches = rep_tensor.shape[0]//batch_size
        rep_tensor = rep_tensor[:num_batches*batch_size]
        rep_tensor = rep_tensor.view(batch_size,-1)
        return rep_tensor

In [4]:
embed_size = 128
hidden_size = 1024 # for individual LSTM units
num_layers = 1 # for stacking LSTMs (by default it is 1) 
num_epochs = 20
batch_size = 20
timesteps = 30
lr = 0.002

In [5]:
corpus = txtprocess()
rep_tensor = corpus.get_data("alice.txt", batch_size)
# this tensor contains the index of all the words in the txt file !
print(rep_tensor.shape) 

torch.Size([20, 1484])


In [7]:
vocab_size = len(corpus.dictionary)
print(vocab_size)
num_batches = rep_tensor.shape[1]//timesteps
print(num_batches)

5290
49


In [8]:
class TextGenerator(nn.Module) :
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers) : 
        super(TextGenerator, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size,hidden_size,num_layers,batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

    # batch_first = True makes the output of lstm layer of size : (batch_size, timesteps, embed_size)

    def forward(self, x, h) : 
        x = self.embed(x)
        # setting batch_first = True is equivalent to
        # x = x.view(batch_size, timesteps, embed_size)
        out, (h,c) = self.lstm(x,h)
        out = out.reshape(out.size(0)*out.size(1),out.size(2))
        out = self.linear(out)
        return out, (h,c)

model = TextGenerator(vocab_size, embed_size, hidden_size, num_layers)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

We will train the model like this !

Suppose we have a string "Black horse is here"

then we as input x we will feed "Black Horse" and expected output we will feed as "lack horse i"
which is basically a one letter delayed input !
similarly next i/o will be "lack horse i"/"ack horse is" and so on... (sliding window kind of thing) !

In [10]:
for epoch in range(num_epochs) : 
    # set hidden and cell (memory) states.
    states = (torch.zeros(num_layers, batch_size, hidden_size),
              torch.zeros(num_layers, batch_size, hidden_size))
    for i in range(0,rep_tensor.size(1)-timesteps, timesteps) : 
        inputs = rep_tensor[:, i:i+timesteps] # black horse
        targets = rep_tensor[:, (i+1):(i+1+timesteps)] # lack horse i
        outputs, _ = model(inputs, states)
        loss = loss_fn(outputs, targets.reshape(-1))
        model.zero_grad()
        loss.backward()
        clip_grad_norm(model.parameters(), 0.5) # maximum allowed gradient value !
        # grads are clipped in the range [-clip_value,clip_value] to prevent exploding gradient problem !
        optimizer.step()
        step = (i+1)//timesteps
        if step%100 == 0 : 
            print(f"epoch {epoch+1}/{num_epochs}, loss = {loss.item():.4f}")

C:\Users\tanma\AppData\Local\Temp\ipykernel_23860\3048611485.py:12: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  clip_grad_norm(model.parameters(), 0.5) # maximum allowed gradient value !


epoch 1/20, loss = 8.5732
epoch 2/20, loss = 5.9295
epoch 3/20, loss = 5.2197
epoch 4/20, loss = 4.7189
epoch 5/20, loss = 4.2329
epoch 6/20, loss = 3.8010
epoch 7/20, loss = 3.3576
epoch 8/20, loss = 2.8335
epoch 9/20, loss = 2.4618
epoch 10/20, loss = 2.0977
epoch 11/20, loss = 1.8720
epoch 12/20, loss = 1.5988
epoch 13/20, loss = 1.2462
epoch 14/20, loss = 0.9097
epoch 15/20, loss = 0.6888
epoch 16/20, loss = 0.4639
epoch 17/20, loss = 0.2862
epoch 18/20, loss = 0.1689
epoch 19/20, loss = 0.1036
epoch 20/20, loss = 0.0836


In [11]:
with torch.no_grad() : 
    with open("results.txt","w") as f : 
        state = (torch.zeros(num_layers, 1, hidden_size),
                 torch.zeros(num_layers, 1, hidden_size))
        inp = torch.randint(0,vocab_size, (1,)).long().unsqueeze(1)
        for i in range(500) : 
            output, _ = model(inp, state)
            print(output.shape)
            prob = output.exp()
            word_id = torch.multinomial(prob, num_samples = 1).item()
            print(word_id)
            inp.fill_(word_id)
            word = corpus.dictionary.idx2word[word_id]
            word = "\n" if word == "<eos>" else word + " "
            f.write(word)

            if (i+1)%100 == 0 : 
                print(f"sampled [{i+1}/500 words")

torch.Size([1, 5290])
5
torch.Size([1, 5290])
6
torch.Size([1, 5290])
129
torch.Size([1, 5290])
114
torch.Size([1, 5290])
44
torch.Size([1, 5290])
4441
torch.Size([1, 5290])
5
torch.Size([1, 5290])
1720
torch.Size([1, 5290])
73
torch.Size([1, 5290])
20
torch.Size([1, 5290])
320
torch.Size([1, 5290])
74
torch.Size([1, 5290])
3
torch.Size([1, 5290])
4772
torch.Size([1, 5290])
5
torch.Size([1, 5290])
3830
torch.Size([1, 5290])
114
torch.Size([1, 5290])
44
torch.Size([1, 5290])
2321
torch.Size([1, 5290])
3
torch.Size([1, 5290])
96
torch.Size([1, 5290])
1274
torch.Size([1, 5290])
167
torch.Size([1, 5290])
9
torch.Size([1, 5290])
3
torch.Size([1, 5290])
5234
torch.Size([1, 5290])
1091
torch.Size([1, 5290])
272
torch.Size([1, 5290])
555
torch.Size([1, 5290])
20
torch.Size([1, 5290])
320
torch.Size([1, 5290])
74
torch.Size([1, 5290])
3
torch.Size([1, 5290])
2065
torch.Size([1, 5290])
5
torch.Size([1, 5290])
898
torch.Size([1, 5290])
781
torch.Size([1, 5290])
372
torch.Size([1, 5290])
395
torch